# Fine-Tuning LLaMA 3.1 For Price Prediction


Fine-tuning an open-source LLM (LLaMA 3.2 3B) for product price estimation using QLoRA.

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [ ]:
# imports
import os
import re
import math
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch


from trl import SFTTrainer, SFTConfig


# Hugging Face libraries
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
from peft import (LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict,
                  prepare_model_for_kbit_training, set_peft_model_state_dict)
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig


# Weights and Biases
import wandb

# Google Colab
from google.colab import userdata



In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.1"
PROJECT_NAME = "price"
HF_USER = "thomasjuma" # your HF name here!

LITE_MODE = True

DATA_USER = "thomasjuma"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

---
### Login to HuggingFace and Weights & Biases

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

---
### Quantisation
The model is "quantized" - we are reducing the precision to 4 bits.

>**QLoRA (Quantized Low-Rank Adapter)** is a method for efficiently fine-tuning large language models >(LLMs) like GPT-3, GPT-4 and LLaMA. These models require high memory and computing power making >training costly for smaller labs. 
>QLoRA combines quantization which reduces model size, with low-rank adapters, small trainable layers >for specific tasks. 
>
>1. Quantization
>
>    Model weights are normally stored in 16-bit or 32-bit floating-point precision which uses a lot >of memory.
>    Quantization reduces this precision to lower-bit formats (e.g 8-bit or 4-bit), shrinking the >model size and speeding up computation.
>    Advanced methods like Normal Float 4-bit (NF4) help maintain accuracy despite the lower precision.
>
>2. Low-Rank Adaptation
>
>    Fine-tuning all weights in a large model is expensive. LoRA adds small trainable adapter layers >to selected parts of the model (attention layers) while keeping the main weights frozen.
>    Only these adapters are updated during fine-tuning, reducing the number of parameters to train.
>    Despite their small size, adapters allow the model to learn task-specific adjustments efficiently.



In [ ]:
# Configure quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

print(f"Quantization: {'4-bit' if QUANT_4_BIT else '8-bit'}")

---
### Load the Tokenizer and the Base Model using 4 bit


In [ ]:
# Load the Tokenizer and the Model

print(f"Loading tokenizer for {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded (vocab size: {len(tokenizer):,})")


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"✓ Base model loaded")
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Prepare model for training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA config object with LoRA Parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
)

# Apply LoRA to model
print("Applying LoRA adapters...")
model = get_peft_model(base_model, lora_parameters)
model.print_trainable_parameters()
print("✓ Model ready for training")

In [ ]:
# Training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)

# Prepare the Trainer

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

---
### Train the model

In [ ]:
# Train
print("="*50)
print("Starting training...")
print("="*50)

train_result = trainer.train()

print("\n" + "="*50)
print("✓ Training complete!")
print(f"  - Final loss: {train_result.metrics['train_loss']:.4f}")
print(f"  - Total samples: {train_result.metrics['total_flos']:.2e} FLOPs")
print("="*50)


In [ ]:
# Push our fine-tuned model to Hugging Face
trainer.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

# Save the fine-tuned model
output_dir = "./price-predictor-finetuned"
print(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✓ Model saved")

if LOG_TO_WANDB:
  wandb.finish()



---
### Load dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
test_data = dataset['test']

---
### Inference and evaluation.

In [ ]:
# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
# Test predictions on sample data
print("=== Testing Predictions ===")
print("\nSample predictions from test set:\n")

for i in range(5):
    sample = test_data[i]
    prompt = sample['prompt']
    actual_price = float(sample['completion'])
    
    predicted = model_predict(prompt)
    error = abs(predicted - actual_price)
    
    print(f"Sample {i+1}:")
    print(f"  Actual: ${actual_price:.2f}")
    print(f"  Predicted: ${predicted:.2f}")
    print(f"  Error: ${error:.2f}")

NB:
1. Cut-off - truncated data
2. Loading the data.
3. For base models, we do not need a system prompt like instruct model. base models are better for single instructions like this one. the instruct models will require more tokens and wont perform better than the base model.
4. 
